# A Universal Identity for Powers in Quadratic Algebras
## and a Matrix Derivation of a Fibonacci Identity

Based on: arxiv.org/abs/2603.19343v1 by Marco Mantovanelli

### Key Results:
1. **Universal Identity**: In a quadratic algebra with basis {1, x} where x² = ax + b,
   any power xⁿ can be expressed as Aₙ·x + Bₙ
2. **Matrix Formula**: For any 2×2 matrix M with trace t and determinant d,
   Mⁿ = αₙ·M + βₙ·I where αₙ, βₙ depend only on t and d
3. **Fibonacci Application**: Applying to the Fibonacci matrix yields a binomial
   expansion formula for Fₙₘ, recovering Vorobtsov's identity

In [ ]:
import numpy as np
from typing import Tuple, List
import math

## Part 1: Quadratic Algebra Framework

A **quadratic algebra** has the form 𝕽[x]/(x² - a·x - b) where:
- Basis: {1, x}
- Multiplication: x · x = a·x + b·1

Any element p·x + q can be raised to any power using the recurrence:
x² = a·x + b
xⁿ⁺¹ = x · xⁿ = Aₙ₊₁·x + Bₙ₊₁
where Aₙ₊₁ = a·Aₙ + Bₙ and Bₙ₊₁ = b·Aₙ

In [ ]:
def quadratic_algebra_power_coeffs(a: float, b: float, n: int) -> Tuple[float, float]:
    """
    Returns coefficients (A_n, B_n) such that x^n = A_n * x + B_n in the
    quadratic algebra defined by x^2 = a*x + b.
    
    These satisfy the recurrence:
    A_0 = 0, B_0 = 1  (x^0 = 1)
    A_1 = 1, B_1 = 0  (x^1 = x)
    A_{n+1} = a*A_n + B_n
    B_{n+1} = b*A_n
    """
    if n == 0:
        return (0.0, 1.0)  # x^0 = 1
    if n == 1:
        return (1.0, 0.0)  # x^1 = x
    
    A_n, B_n = 1.0, 0.0  # A_1, B_1
    A_prev, B_prev = 0.0, 1.0  # A_0, B_0
    
    for k in range(2, n + 1):
        A_new = a * A_n + B_n
        B_new = b * A_n
        A_prev, B_prev = A_n, B_n
        A_n, B_n = A_new, B_new
    
    return (A_n, B_n)


def verify_quadratic_identity(a: float, b: float, n: int) -> bool:
    """
    Verify that x^n = A_n*x + B_n in the algebra.
    We do this symbolically by checking the recurrence.
    """
    A_n, B_n = quadratic_algebra_power_coeffs(a, b, n)
    
    # For n=2: x^2 = a*x + b, so A_2 = a, B_2 = b
    if n == 2:
        return abs(A_n - a) < 1e-10 and abs(B_n - b) < 1e-10
    
    # For n >= 3, verify recurrence: A_n = a*A_{n-1} + B_{n-1}, B_n = b*A_{n-1}
    A_prev, B_prev = quadratic_algebra_power_coeffs(a, b, n-1)
    A_check = a * A_prev + B_prev
    B_check = b * A_prev
    
    return abs(A_n - A_check) < 1e-10 and abs(B_n - B_check) < 1e-10


# Test the quadratic algebra identity
print("Testing quadratic algebra identity...")
for a, b in [(1, 1), (0, -1), (3, 2), (1, -1)]:
    for n in range(2, 8):
        A_n, B_n = quadratic_algebra_power_coeffs(a, b, n)
        valid = verify_quadratic_identity(a, b, n)
        print(f"  a={a}, b={b}, n={n}: x^{n} = {A_n:.4f}·x + {B_n:.4f} | Valid: {valid}")

## Part 2: 2×2 Matrix Power Formula

**Theorem (Cayley-Hamilton)**: Every 2×2 matrix M satisfies:
M² = t·M - d·I
where t = trace(M) and d = det(M)

This gives us a recurrence for Mⁿ:
Mⁿ = αₙ·M + βₙ·I

with:
- α₀ = 0, β₀ = 1 (M⁰ = I)
- α₁ = 1, β₁ = 0 (M¹ = M)
- α_{n+1} = t·αₙ - d·α_{n-1}
- β_{n+1} = t·βₙ - d·β_{n-1}

In [ ]:
def matrix_power_coeffs(trace_m: float, det_m: float, n: int) -> Tuple[float, float]:
    """
    Returns coefficients (alpha_n, beta_n) such that M^n = alpha_n * M + beta_n * I
    for a 2x2 matrix M with given trace and determinant.
    
    These satisfy:
    alpha_0 = 0, beta_0 = 1
    alpha_1 = 1, beta_1 = 0
    alpha_{n+1} = t * alpha_n - d * alpha_{n-1}
    beta_{n+1} = t * beta_n - d * beta_{n-1}
    
    This is analogous to the recurrence for Fibonacci numbers.
    """
    if n == 0:
        return (0.0, 1.0)  # M^0 = I
    if n == 1:
        return (1.0, 0.0)  # M^1 = M
    
    alpha_n, beta_n = 1.0, 0.0   # alpha_1, beta_1
    alpha_prev, beta_prev = 0.0, 1.0  # alpha_0, beta_0
    
    for k in range(2, n + 1):
        alpha_new = trace_m * alpha_n - det_m * alpha_prev
        beta_new = trace_m * beta_n - det_m * beta_prev
        alpha_prev, beta_prev = alpha_n, beta_n
        alpha_n, beta_n = alpha_new, beta_new
    
    return (alpha_n, beta_n)


def compute_matrix_power(M: np.ndarray, n: int) -> np.ndarray:
    """
    Compute M^n using the trace-determinant formula.
    Validates against numpy's matrix power.
    """
    t = np.trace(M)
    d = np.linalg.det(M)
    alpha_n, beta_n = matrix_power_coeffs(t, d, n)
    
    M_n_formula = alpha_n * M + beta_n * np.eye(M.shape[0])
    M_n_direct = np.linalg.matrix_power(M, n)
    
    return M_n_formula, M_n_direct, np.allclose(M_n_formula, M_n_direct)


# Test with random matrices
print("\nTesting 2x2 matrix power formula...")
test_matrices = [
    np.array([[1, 1], [1, 0]], dtype=float),  # Fibonacci matrix
    np.array([[2, 1], [1, 2]], dtype=float),  # Symmetric matrix
    np.array([[3, -1], [1, 1]], dtype=float),  # General matrix
]

for M in test_matrices:
    t, d = np.trace(M), np.linalg.det(M)
    print(f"\nMatrix M =\n{M}")
    print(f"  trace = {t}, det = {d}")
    for n in range(5):
        M_formula, M_direct, match = compute_matrix_power(M, n)
        print(f"  M^{n}: formula matches direct = {match}")

## Part 3: Fibonacci Matrix Application

The **Fibonacci matrix** is:
F = [[1, 1], [1, 0]]

Properties:
- det(F) = -1
- trace(F) = 1
- Fⁿ = [[F_{n+1}, F_n], [F_n, F_{n-1}]] (well-known identity)

Using our formula with t=1, d=-1:
α_{n+1} = 1·αₙ - (-1)·α_{n-1} = αₙ + α_{n-1}
β_{n+1} = 1·βₙ - (-1)·β_{n-1} = βₙ + β_{n-1}

These are **Fibonacci recurrences**!

The result is the **binomial expansion formula for F_{nm}**:
F_{nm} = Σ_{k=0}^{⌊(nm-1)/2⌋} C(nm, 2k+1) · F_{k+1} · (-1)^k

This recovers **Vorobtsov's identity**.

In [ ]:
# Fibonacci numbers
def fib(n: int) -> int:
    """Compute nth Fibonacci number (F_0=0, F_1=1, F_2=1, ...)"""
    if n <= 0:
        return 0
    if n == 1:
        return 1
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


# Fibonacci matrix power
FIB_MATRIX = np.array([[1, 1], [1, 0]], dtype=float)

print("Fibonacci matrix F = [[1,1],[1,0]]:")
print(f"  trace = {np.trace(FIB_MATRIX)}, det = {np.linalg.det(FIB_MATRIX)}")

# Verify F^n = [[F_{n+1}, F_n], [F_n, F_{n-1}]]
print("\nVerifying F^n = [[F_{n+1}, F_n], [F_n, F_{n-1}]]:")
for n in range(1, 10):
    F_n = np.linalg.matrix_power(FIB_MATRIX, n)
    expected = np.array([[fib(n+1), fib(n)], [fib(n), fib(n-1)]], dtype=float)
    match = np.allclose(F_n, expected)
    print(f"  F^{n}: computed = {F_n[0,0]}, expected F_{n+1}={fib(n+1)} | Match: {match}")

In [ ]:
# The binomial expansion formula for F_{nm}
# This is Vorobtsov's identity, derived from the general matrix formula

def vorobtsov_fib_identity(n: int, m: int) -> int:
    """
    Compute F_{nm} using the binomial/Vorobtsov identity:
    F_{nm} = sum_{k=0}^{floor((nm-1)/2)} C(nm, 2k+1) * F_{k+1} * (-1)^k
    
    This is derived from the universal quadratic algebra identity
    applied to the Fibonacci matrix.
    """
    nm = n * m
    result = 0
    max_k = (nm - 1) // 2
    
    for k in range(max_k + 1):
        # C(nm, 2k+1) * F_{k+1} * (-1)^k
        binomial = math.comb(nm, 2*k + 1)
        term = binomial * fib(k + 1) * ((-1) ** k)
        result += term
    
    return result


print("Vorobtsov's Identity: F_{nm} = sum_{k=0}^{floor((nm-1)/2)} C(nm, 2k+1) * F_{k+1} * (-1)^k")
print("\nVerifying against direct Fibonacci computation:\n")

test_cases = [(2, 3), (3, 4), (4, 5), (5, 6), (3, 7), (2, 8), (6, 6)]
for n, m in test_cases:
    f_nm_direct = fib(n * m)
    f_nm_vorobtsov = vorobtsov_fib_identity(n, m)
    match = f_nm_direct == f_nm_vorobtsov
    print(f"  F_{n}*{m} = F_{n*m}: direct={f_nm_direct}, vorobtsov={f_nm_vorobtsov} | Match: {match}")

## Part 4: Deriving the Universal Formula

### The General Pattern

For any quadratic algebra element x with x² = ax + b:

xⁿ = **U**ₙ(a,b) · x + **V**ₙ(a,b)

where **U**ₙ and **V**ₙ are universal polynomials in a and b.

For matrices, the same pattern holds with trace t and determinant d:

Mⁿ = **α**ₙ(t,d) · M + **β**ₙ(t,d) · I

The coefficients satisfy Fibonacci-like recurrences:
- **U**_{n+1} = a·**U**ₙ + **V**ₙ, **V**_{n+1} = b·**U**ₙ
- **α**_{n+1} = t·**α**ₙ - d·**α**_{n-1}, **β**_{n+1} = t·**β**ₙ - d·**β**_{n-1}

### Key Insight from the Paper

These identities are **not coincidences** - they arise from the general
algebraic structure of quadratic algebras. The Fibonacci identities are
special cases of a universal principle.

In [ ]:
# Display the connection between algebraic structure and Fibonacci
print("=" * 70)
print("SUMMARY: Universal Identity in Quadratic Algebras")
print("=" * 70)
print("""
The paper proves that for ANY quadratic algebra (basis {{1, x}} with x² = ax + b):

    x^n = U_n(a,b) · x + V_n(a,b)

For 2×2 matrices with trace t and det d:

    M^n = α_n(t,d) · M + β_n(t,d) · I

The Fibonacci matrix F = [[1,1],[1,0]] has t=1, d=-1.
The recurrence α_{n+1} = α_n + α_{n-1} produces Fibonacci numbers!

The binomial formula for F_{{nm}}:
    F_{{nm}} = Σ C(nm, 2k+1) · F_{{k+1}} · (-1)^k

This is Vorobtsov's identity, now derived from general principles.
""")

# Show explicit coefficients for Fibonacci matrix
print("\nExplicit coefficients for M^n = α_n · M + β_n · I where M = Fibonacci matrix:")
t, d = 1, -1
for n in range(8):
    alpha, beta = matrix_power_coeffs(t, d, n)
    print(f"  n={n}: α_{n}={alpha:6.2f}, β_{n}={beta:6.2f}")

In [ ]:
# Final verification with symbolic derivation
print("\n" + "=" * 70)
print("VERIFICATION: Full Matrix Power Computation")
print("=" * 70)

for n in [5, 10, 15, 20]:
    # Using our formula
    alpha, beta = matrix_power_coeffs(1, -1, n)
    M_formula = alpha * FIB_MATRIX + beta * np.eye(2)
    
    # Direct computation
    M_direct = np.linalg.matrix_power(FIB_MATRIX, n)
    
    # Extract F_{n+1} from the top-left element
    F_n_plus_1_formula = int(round(M_formula[0, 0]))
    F_n_plus_1_direct = int(round(M_direct[0, 0]))
    
    print(f"n={n:2d}: α={alpha:7.2f}, β={beta:7.2f} | F_{n+1}={F_n_plus_1_direct:7d} | Match: {F_n_plus_1_formula == F_n_plus_1_direct}")